# YOLOv12 — Road Damage Detection

Trains an Ultralytics **YOLOv12** model on the Pothole detection dataset for the Road Damage Classification, Detection, and Reporting pipeline.

Mirrors the structure of `YOLOv8.ipynb` so results are directly comparable.

**Recommended runtime:** Google Colab with GPU (T4 or better).

## 1. Install dependencies

In [ ]:
!pip install -q "ultralytics>=8.3.0" roboflow

In [ ]:
import os
import ultralytics
ultralytics.checks()

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. Imports & workspace

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image

HOME = os.getcwd()
print('HOME:', HOME)

## 3. Download dataset from Roboflow

The Roboflow API key is read from the `ROBOFLOW_API_KEY` environment variable. In Colab set it via:

```python
import os
os.environ['ROBOFLOW_API_KEY'] = 'your_key_here'
```

or use Colab's *Secrets* panel.

In [ ]:
from roboflow import Roboflow

api_key = os.environ.get('ROBOFLOW_API_KEY')
if not api_key:
    raise RuntimeError('Set ROBOFLOW_API_KEY before running this cell.')

rf = Roboflow(api_key=api_key)
project = rf.workspace('khan-hhcjg').project('pothole-fyv8h')
version = project.version(1)
dataset = version.download('yolov8')  # YOLOv12 uses the same label format as YOLOv8
print('Dataset downloaded to:', dataset.location)

In [ ]:
!ls {dataset.location}

## 4. Load YOLOv12 base model

Ultralytics auto-downloads the pretrained weights on first use.

In [ ]:
model = YOLO('yolo12n.pt')

In [ ]:
dataset_path = Path(dataset.location) / 'data.yaml'
assert dataset_path.exists(), f'data.yaml not found at {dataset_path}'
print('Using dataset config at:', dataset_path.resolve())

## 5. Train

In [ ]:
results = model.train(
    data=str(dataset_path),
    epochs=500,
    imgsz=640,
    batch=16,
    name='road_damage_detection_yolov12',
)
print('Training completed.')

## 6. Validate on the held-out split

In [ ]:
metrics = model.val()
print(metrics)

## 7. Run inference on validation samples

In [ ]:
valid_images = Path(dataset.location) / 'valid' / 'images'
preds = model.predict(
    source=str(valid_images),
    conf=0.25,
    save=True,
    name='road_damage_detection_yolov12_predict',
)
print(f'Ran inference on {len(preds)} images.')

## 8. Export the trained weights

The best checkpoint is written by Ultralytics to `runs/detect/road_damage_detection_yolov12/weights/best.pt`. Copy it into the Django backend (set `MODEL_PATH2` in `.env`) to serve YOLOv12 in production.

In [ ]:
best_weights = Path('runs/detect/road_damage_detection_yolov12/weights/best.pt')
print('Best weights:', best_weights.resolve() if best_weights.exists() else 'not found yet')